# Can we predict failure within 90 days of an inspection?
Target: `failed_90d`. Plan: inspect data -> check for leakage -> time-based split -> compare against baselines -> report honestly.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width', 140)
df = pd.read_csv('data.csv', parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(df.shape, 'missing:', int(df.isna().sum().sum()), 'dup rows:', int(df.duplicated().sum()), 'unique assets:', df.asset_tag.nunique())
print('base rate:', round(df.failed_90d.mean(), 4), '| span:', df.inspected_at.min(), '->', df.inspected_at.max())
df.describe().T

## 1. Leakage check: `parts_replaced`
The dictionary says it is the number of parts replaced on the work order *that closed this inspection record* - it is recorded after the inspection and plausibly as a result of a failure/repair. Compare with the target:

In [ ]:
print(df.groupby('parts_replaced').failed_90d.agg(['mean', 'size']))

Every record with >=2 parts replaced failed, and none with 0 did. That is not a signal available at inspection time; it is the outcome leaking into the feature. **It is excluded from all models.** (Worth confirming with the work-order data owner when the order is closed relative to the failure.)

In [ ]:
FEATS_NUM = ['vibration_rms', 'bearing_temp_c', 'ambient_temp_c', 'meter_hours', 'load_pct']
FEATS_CAT = ['site']
X = FEATS_NUM + FEATS_CAT
print('AUC of parts_replaced alone (leaky, NOT used):', round(roc_auc_score(df.failed_90d, df.parts_replaced), 3))

## 2. Exploration

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(14, 7))
for a, f in zip(ax.flat, FEATS_NUM):
    for y, col in [(0, 'tab:blue'), (1, 'tab:red')]:
        a.hist(df.loc[df.failed_90d == y, f], bins=40, alpha=.55, density=True, color=col, label=f'failed={y}')
    a.set_title(f); a.legend()
q = df.groupby(df.inspected_at.dt.to_period('Q')).failed_90d.mean()
ax.flat[5].bar(q.index.astype(str), q.values); ax.flat[5].set_title('failure rate by inspection quarter'); ax.flat[5].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()
print(df.groupby('site').failed_90d.agg(['mean', 'size']))
print(df[FEATS_NUM + ['failed_90d']].corr()['failed_90d'].round(3))

The failure rate swings strongly over time (about 4% -> 27% -> 3%). Causes could be real events/seasonality, or right-censoring of the latest inspections. Either way a random split may be optimistic, so a **chronological split is the headline**.

## 3. Models - chronological hold-out (train on first 70%, test on last 30%)

In [ ]:
cut = int(len(df) * .7)
tr, te = df.iloc[:cut], df.iloc[cut:]
print('train base rate', round(tr.failed_90d.mean(), 3), '| test base rate', round(te.failed_90d.mean(), 3))
pre = ColumnTransformer([('n', StandardScaler(), FEATS_NUM), ('c', OneHotEncoder(), FEATS_CAT)])
models = {'logistic': make_pipeline(pre, LogisticRegression(max_iter=1000)),
          'grad_boost': make_pipeline(pre, HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=200, random_state=0))}
rows = []; probs = {}
for n, m in models.items():
    m.fit(tr[X], tr.failed_90d); p = m.predict_proba(te[X])[:, 1]; probs[n] = p
    rows.append((n, roc_auc_score(te.failed_90d, p), average_precision_score(te.failed_90d, p)))
rows.append(('no-skill (base rate)', .5, te.failed_90d.mean()))
pd.DataFrame(rows, columns=['model', 'ROC AUC', 'PR AUC']).round(3)

### Cross-check: stratified 5-fold CV, plus a shuffled-label control

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for n, m in models.items():
    p = cross_val_predict(m, df[X], df.failed_90d, cv=cv, method='predict_proba')[:, 1]
    print(f'{n}: CV AUC={roc_auc_score(df.failed_90d, p):.3f}  AP={average_precision_score(df.failed_90d, p):.3f}')
rng = np.random.default_rng(0)
y_shuf = rng.permutation(df.failed_90d.values)
p = cross_val_predict(models['grad_boost'], df[X], y_shuf, cv=cv, method='predict_proba')[:, 1]
print('shuffled-label control AUC:', round(roc_auc_score(y_shuf, p), 3), '(should be ~0.5)')

In [ ]:
best = max(probs, key=lambda n: roc_auc_score(te.failed_90d, probs[n]))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for n, p in probs.items():
    f, t, _ = roc_curve(te.failed_90d, p); ax[0].plot(f, t, label=n)
    pr, rc, _ = precision_recall_curve(te.failed_90d, p); ax[1].plot(rc, pr, label=n)
ax[0].plot([0, 1], [0, 1], 'k--'); ax[0].set(title='ROC (test)', xlabel='FPR', ylabel='TPR'); ax[0].legend()
ax[1].axhline(te.failed_90d.mean(), color='k', ls='--'); ax[1].set(title='Precision-Recall (test)', xlabel='recall', ylabel='precision'); ax[1].legend()
plt.tight_layout(); plt.show()
d = pd.DataFrame({'y': te.failed_90d.values, 'p': probs[best]})
d['decile'] = pd.qcut(d.p.rank(method='first'), 10, labels=False)
print('Failure rate by predicted-risk decile (best model, test):'); print(d.groupby('decile').y.mean().round(3).to_string())

## 4. What drives the prediction?

In [ ]:
r = permutation_importance(models[best], te[X], te.failed_90d, scoring='roc_auc', n_repeats=20, random_state=0)
print(pd.Series(r.importances_mean, index=X).sort_values(ascending=False).round(4))

## 5. Summary

In [ ]:
print('Best model:', best)
print('Test ROC AUC:', round(roc_auc_score(te.failed_90d, probs[best]), 3), '| PR AUC:', round(average_precision_score(te.failed_90d, probs[best]), 3), '| base rate:', round(te.failed_90d.mean(), 3))

Caveats: `parts_replaced` excluded as leakage; failure rate varies strongly over time (possible censoring of late inspections); one inspection per asset, so no per-asset history features were possible.